In [ ]:
################
#FT-Transformer Preprocessing

#Must convert to parquet to be memory efficient given the low memory env in datahub

#Optimal features (19 ):
#  education_tier, MSP, WKHP, ESR, ENG, CIT, MAR, MIG, is_latinx, has_insurance,
#  AGEP, OCCP, race_white, race_black, race_asian, race_indigenous, race_other,
#  race_ethnic_aggregate, Sex


###############

In [1]:
import pandas as pd
import json
import os
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import resample

In [2]:
OUT_DIR    = 'preprocessing_data'     
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
df_train = pd.read_csv('preprocessing_data/train_engineered.csv')
df_test  = pd.read_csv('preprocessing_data/test_engineered.csv')

In [4]:
print(f"  Train: {df_train.shape}  |  Test: {df_test.shape}")
print(f"  Train binary dist: {df_train['binary_target'].value_counts().to_dict()}")
print(f"  Test  binary dist: {df_test['binary_target'].value_counts().to_dict()}")


  Train: (1469769, 58)  |  Test: (304368, 58)
  Train binary dist: {0: 1114746, 1: 355023}
  Test  binary dist: {0: 233793, 1: 70575}


In [5]:
with open('preprocessing_data/feature_engineering_metadata.json') as f:
    meta = json.load(f)


In [6]:
optimal_features = meta['optimal_features'] 
categorical_features = meta['categorical_features']

optimal_cat  = [f for f in optimal_features if f in categorical_features]
optimal_cont = [f for f in optimal_features if f not in categorical_features]

print(f"\nOptimal features ({len(optimal_features)}): {optimal_features}")
print(f"  Categorical ({len(optimal_cat)}): {optimal_cat}")
print(f"  Continuous  ({len(optimal_cont)}): {optimal_cont}")


Optimal features (21): ['WKL', 'education_tier', 'MSP', 'WKHP', 'LANX', 'ESR', 'MAR', 'ENG', 'is_latinx', 'CIT', 'MIG', 'has_insurance', 'AGEP', 'SEX', 'OCCP', 'race_white', 'race_black', 'race_asian', 'race_indigenous', 'race_other', 'race_ethnic_aggregate']


NameError: name 'optimal_cat' is not defined

In [ ]:
#downsample

df_stable  = df_train[df_train['poverty_risk_score'] == 0]
df_at_risk = df_train[df_train['poverty_risk_score'] > 0]
df_stable_down = resample(
    df_stable, 
    replace=False,
    n_samples=len(df_at_risk),
    random_state=42)
train_4class = pd.concat([df_stable_down, df_at_risk]).sample(frac=1, random_state=42)


In [ ]:
print(f"\n4-class balanced distribution:")
print(train_4class['poverty_risk_score'].value_counts().sort_index())

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_bal, y_bal = rus.fit_resample(
    df_train[optimal_features],
    df_train['binary_target']
)
train_binary = pd.DataFrame(X_bal, columns=optimal_features)
train_binary['binary_target'] = y_bal.values

In [ ]:
#saving to parquet 

train_4class_path = os.path.join(OUT_DIR, 'ft_train_4class.parquet')
train_binary_path = os.path.join(OUT_DIR, 'ft_train_binary.parquet')
test_path         = os.path.join(OUT_DIR, 'ft_test.parquet')
meta_path         = os.path.join(OUT_DIR, 'ft_metadata.json')

cols_4class = optimal_features + ['binary_target', 'poverty_risk_score']
cols_test   = optimal_features + ['binary_target', 'poverty_risk_score']

train_4class[cols_4class].to_parquet(train_4class_path, index=False)
train_binary.to_parquet(train_binary_path, index=False)
df_test[cols_test].to_parquet(test_path, index=False)
